# 01 — Setup & Parse SUN RGB-D Scenes

**Purpose:** One-time environment bootstrap for the IKEA interior-design generation project.

This notebook:
1. Mounts Google Drive and wires up the project directory tree.
2. Clones the repo and installs Python dependencies.
3. Copies SUN RGB-D metadata from Drive into the working directory.
4. Runs `src.data_parser` to produce `data/processed/structured_scenes.json`.
5. Shows a summary table of the parsed scenes.
6. Downloads reference RGB images via `src.utils.download_sunrgbd_sample`.
7. Commits the processed JSON back to Drive for session persistence.

**Runtime required:** CPU is sufficient for this notebook. Switch to a T4 GPU runtime before opening notebook 02.

---

## Cell 1 — Mount Drive & Create Project Directories

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/ikea-sd")

SUBDIRS = [
    DRIVE_ROOT / "data" / "raw",
    DRIVE_ROOT / "data" / "metadata",
    DRIVE_ROOT / "data" / "processed",
    DRIVE_ROOT / "outputs" / "generated_images",
    DRIVE_ROOT / "outputs" / "grids",
    DRIVE_ROOT / "report" / "figures",
]

for d in SUBDIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Drive mounted. Project subdirectories:")
for d in SUBDIRS:
    print(f"  {d}")

## Cell 2 — Clone Repo

In [ ]:
import os
from pathlib import Path

# ── EDIT THIS LINE: replace with your actual GitHub repo URL ──────────────
REPO_URL = "https://github.com/YOUR_USERNAME/ikea-sd.git"
# ─────────────────────────────────────────────────────────────────────────

REPO_DIR = Path("/content/ikea-sd")

if REPO_DIR.is_dir():
    print(f"Repo already exists at {REPO_DIR} — pulling latest changes.")
    os.system(f"git -C {REPO_DIR} pull --ff-only")
else:
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError(
            f"git clone failed (exit {ret}). "
            "Check that REPO_URL is set correctly above."
        )

print(f"\nRepo ready at {REPO_DIR}")
print("Contents:", sorted(p.name for p in REPO_DIR.iterdir()))

## Cell 3 — Install Dependencies

In [ ]:
%cd /content/ikea-sd

# torch and torchvision are pre-installed on Colab T4 — skip them to avoid
# breaking the CUDA environment. Install everything else from requirements.txt.
import subprocess, sys

with open("requirements.txt") as f:
    reqs = [
        line.strip()
        for line in f
        if line.strip()
        and not line.startswith("#")
        and not line.lower().startswith("torch")
        and not line.lower().startswith("numpy")
        and not line.lower().startswith("pandas")
    ]

print(f"Installing {len(reqs)} packages (torch/torchvision/numpy/pandas excluded)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *reqs],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("pip install failed — see stderr above.")

print("Installation complete.")

## Cell 4 — Copy SUN RGB-D Metadata from Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ikea-sd")
LOCAL_RAW  = Path("/content/ikea-sd/data/raw")
LOCAL_RAW.mkdir(parents=True, exist_ok=True)

# Files to mirror from Drive → local working directory
METADATA_FILES = [
    "SUNRGBDMeta2DBB_v2.mat",
    "SUNRGBDMeta3DBB_v2.mat",  # optional — skip if absent
]

copied, skipped, missing = [], [], []

for fname in METADATA_FILES:
    src = DRIVE_ROOT / "data" / "raw" / fname
    dst = LOCAL_RAW / fname
    if dst.is_file():
        skipped.append(fname)
    elif src.is_file():
        shutil.copy2(src, dst)
        copied.append(fname)
    else:
        missing.append(fname)

print(f"Copied  : {copied  or '—'}")
print(f"Skipped : {skipped or '—'}  (already present)")
print(f"Missing : {missing or '—'}  (not on Drive — run Cell 7 to fetch)")

# Also symlink the SUNRGBD/ folder from Drive to avoid duplicating 6.4 GB
drive_sunrgbd = DRIVE_ROOT / "data" / "raw" / "SUNRGBD"
local_sunrgbd = LOCAL_RAW / "SUNRGBD"

if drive_sunrgbd.is_dir() and not local_sunrgbd.exists():
    local_sunrgbd.symlink_to(drive_sunrgbd)
    print(f"\nSymlinked {local_sunrgbd} → {drive_sunrgbd}")
elif local_sunrgbd.exists():
    print(f"\nSUNRGBD symlink/folder already exists at {local_sunrgbd}")
else:
    print(
        f"\nSUNRGBD/ not found on Drive at {drive_sunrgbd}. "
        "Follow instructions printed by Cell 7 to download it."
    )

## Cell 5 — Parse Scenes (limit 50 for fast iteration)

In [ ]:
import logging
import sys
from pathlib import Path

# Ensure the repo root is on the path regardless of %cd state
REPO_DIR = Path("/content/ikea-sd")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s  %(name)s  %(message)s",
    force=True,
)

from src.data_parser import DataParser, write_json

RAW_ROOT       = REPO_DIR / "data" / "raw"
OUT_JSON       = REPO_DIR / "data" / "processed" / "structured_scenes.json"
PARSE_LIMIT    = 50   # increase to None to parse the full dataset

parser  = DataParser(raw_root=RAW_ROOT, limit=PARSE_LIMIT)
records = parser.load()

write_json(records, OUT_JSON)
print(f"\nParsed {len(records)} scenes → {OUT_JSON}")

## Cell 6 — Summary Table

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

OUT_JSON = Path("/content/ikea-sd/data/processed/structured_scenes.json")

with OUT_JSON.open(encoding="utf-8") as f:
    scenes = json.load(f)

df = pd.DataFrame(scenes)

# ── Summary table ─────────────────────────────────────────────────────────
summary = (
    df.groupby("room_type")
    .agg(
        count=("scene_id", "count"),
        mean_objects=("num_objects", "mean"),
        has_layout=("layout_dims", lambda s: s.notna().sum()),
    )
    .sort_values("count", ascending=False)
    .reset_index()
)
summary["mean_objects"] = summary["mean_objects"].round(1)
summary["pct"] = (100 * summary["count"] / summary["count"].sum()).round(1)

print(f"Total scenes  : {len(df)}")
print(f"Room types    : {df['room_type'].nunique()}")
print(f"With layout   : {df['layout_dims'].notna().sum()}")
print(f"Missing layout: {df['layout_dims'].isna().sum()}")
print()
display(summary)

# ── Bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(summary["room_type"], summary["count"], color="steelblue")
ax.set_xlabel("Scene count")
ax.set_title("Scenes per room type")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Cell 7 — Fetch Reference RGB Images

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/ikea-sd")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils import download_sunrgbd_sample

RAW_ROOT   = REPO_DIR / "data" / "raw"
N_SCENES   = 50   # match the parse limit above

# download_sunrgbd_sample will:
#   1. Fetch SUNRGBDMeta2DBB_v2.mat (~4.3 MB) if not already present.
#   2. Detect whether a pre-extracted SUNRGBD/ tree already exists.
#   3. If not, print clear wget/unzip instructions — no silent failure.
download_sunrgbd_sample(out_dir=RAW_ROOT, n_scenes=N_SCENES)

## Cell 8 — Persist Processed JSON to Drive

In [ ]:
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ikea-sd")
LOCAL_JSON = Path("/content/ikea-sd/data/processed/structured_scenes.json")
DRIVE_JSON = DRIVE_ROOT / "data" / "processed" / "structured_scenes.json"

if not LOCAL_JSON.is_file():
    raise FileNotFoundError(
        f"Expected output not found: {LOCAL_JSON}\n"
        "Re-run Cell 5 to generate it."
    )

DRIVE_JSON.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOCAL_JSON, DRIVE_JSON)

size_kb = LOCAL_JSON.stat().st_size / 1024
print(f"Copied {LOCAL_JSON.name}  ({size_kb:.1f} KB)  →  {DRIVE_JSON}")
print("Processed JSON will survive session restarts via Drive.")

---

## Next Steps

**Proceed to notebook 02 with GPU runtime.**

Before opening `02_generate.ipynb`:
1. Go to **Runtime → Change runtime type → T4 GPU** and reconnect.
2. Confirm `structured_scenes.json` was written to Drive (Cell 8 output above).
3. If the `SUNRGBD/` folder is not yet on Drive, follow the `wget` instructions printed by Cell 7 before generating — the generator needs the reference images for ControlNet conditioning.